# 05: Independent Score Export and Verified Figure Generation
This standalone notebook loads all trained model checkpoints and test datasets from previous runs, performs test-set inference without modifying any prior notebooks, exports per-image y_true and y_score vectors to scores/{backbone}__{setting}.npz, and produces 100% verified publication figures directly from raw model outputs.

In [ ]:
import os, io, json, time, random, glob
from pathlib import Path
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from datasets import load_from_disk
from PIL import Image
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, roc_curve, precision_recall_curve, average_precision_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device, "|", torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU")

def find_dirs(top, names, max_depth=12):
    found = []
    for root, dirs, _ in os.walk(top):
        for d in list(dirs):
            if d.lower() in names: found.append(os.path.join(root, d))
    return found

# Locate datasets
saved = find_dirs("/kaggle/input", {"defactify_subset"})
if not saved: saved = find_dirs("/kaggle/working", {"defactify_subset"})
assert saved, "Attach defactify_subset from 01-ai-fake-image-detector"
SAVED_ROOT = os.path.dirname(saved[0])
SPLITS_DIR = f"{SAVED_ROOT}/splits"
defactify = load_from_disk(f"{SAVED_ROOT}/defactify_subset")

bg_found = find_dirs("/kaggle/input", {"ai", "nature"})
BIGGAN_DIR = os.path.commonpath(bg_found) if bg_found else ""
while os.path.basename(BIGGAN_DIR).lower() in {"ai", "nature", "train", "val"}:
    BIGGAN_DIR = os.path.dirname(BIGGAN_DIR)

print("SPLITS_DIR:", SPLITS_DIR)
print("BIGGAN_DIR:", BIGGAN_DIR)
os.makedirs("scores", exist_ok=True)
os.makedirs("figures", exist_ok=True)


In [ ]:
# Dataset definition identical to training pipeline
IMG_COL = "Image"
class JPEGRecompress:
    def __init__(self, qmin=90, qmax=90):
        self.qmin, self.qmax = qmin, qmax
    def __call__(self, img):
        q = random.randint(self.qmin, self.qmax)
        buf = io.BytesIO()
        img.save(buf, format="JPEG", quality=q)
        buf.seek(0)
        return Image.open(buf).convert("RGB")

class UnifiedImageDataset(Dataset):
    def __init__(self, csv_path, hf_dict, local_dir, transform=None):
        self.df = pd.read_csv(csv_path)
        self.hf_dict = hf_dict
        self.local_dir = local_dir
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        if r.storage == "hf":
            img = self.hf_dict[r.hf_split][int(r.hf_index)][IMG_COL]
        else:
            img = Image.open(os.path.join(self.local_dir, r.image_path)).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, int(r.binary_label)

norm = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
eval_tf = transforms.Compose([
    transforms.CenterCrop(128),
    JPEGRecompress(90, 90),
    transforms.ToTensor(), norm
])

def make_loader(csv, tf, batch_size=128):
    ds = UnifiedImageDataset(csv, defactify, BIGGAN_DIR, transform=tf)
    return DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)


In [ ]:
# Model Builders
def build_resnet50(num_classes=2):
    m = models.resnet50(weights=None)
    m.fc = nn.Linear(m.fc.in_features, num_classes)
    return m

def build_convnext(num_classes=2):
    m = models.convnext_tiny(weights=None)
    m.classifier[2] = nn.Linear(m.classifier[2].in_features, num_classes)
    return m

@torch.no_grad()
def extract_scores(model, loader):
    model.eval()
    y_all, p_all = [], []
    for x, y in loader:
        lo = model(x.to(device))
        probs = torch.softmax(lo.float(), dim=1)[:, 1].cpu().numpy()
        p_all.append(probs)
        y_all.append(y.numpy())
    return np.concatenate(y_all).astype(np.int8), np.concatenate(p_all).astype(np.float32)

def save_scores(setting, backbone, y_true, y_score):
    out_file = f"scores/{backbone}__{setting}.npz"
    np.savez_compressed(out_file, y_true=y_true, y_score=y_score)
    auc = roc_auc_score(y_true, y_score)
    bal_acc = balanced_accuracy_score(y_true, y_score > 0.5)
    print(f"Saved {out_file} | AUC: {auc:.4f} | Bal Acc: {bal_acc*100:.2f}%")
    return {"setting": setting, "backbone": backbone, "auc": float(auc), "bal_acc": float(bal_acc)}


In [ ]:
# Find all checkpoints
ckpt_files = glob.glob("/kaggle/input/**/*.pt", recursive=True)
print(f"Found {len(ckpt_files)} checkpoint files")

def find_ckpt(pattern):
    matches = [f for f in ckpt_files if pattern in os.path.basename(f)]
    return matches[0] if matches else None

settings = [
    "seen_all", "gan_to_diffusion", "diffusion_to_gan",
    "unseen_BigGAN", "unseen_DALL_E_3", "unseen_Midjourney_v6",
    "unseen_Stable_Diffusion_2_1", "unseen_Stable_Diffusion_3", "unseen_Stable_Diffusion_XL"
]

summary_records = []
for s in settings:
    csv_path = f"{SPLITS_DIR}/{s}/test.csv"
    if not os.path.exists(csv_path):
        print("Missing split:", csv_path); continue
    loader = make_loader(csv_path, eval_tf)
    
    # 1. ResNet-50 Fine-tuned
    rn_ckpt = find_ckpt(f"{s}_finetune_A_best.pt")
    if rn_ckpt:
        m_rn = build_resnet50(2).to(device)
        m_rn.load_state_dict(torch.load(rn_ckpt, map_location=device))
        y_true, y_score = extract_scores(m_rn, loader)
        rec = save_scores(s, "resnet50_finetuned", y_true, y_score)
        summary_records.append(rec)
    
    # 2. ConvNeXt-Tiny
    cn_ckpt = find_ckpt(f"convnext_{s}_best.pt")
    if cn_ckpt:
        m_cn = build_convnext(2).to(device)
        m_cn.load_state_dict(torch.load(cn_ckpt, map_location=device))
        y_true, y_score = extract_scores(m_cn, loader)
        rec = save_scores(s, "convnext_tiny", y_true, y_score)
        summary_records.append(rec)

df_sum = pd.DataFrame(summary_records)
df_sum.to_csv("scores/summary_scores_benchmark.csv", index=False)
print("\n--- Completed Binary Score Extraction ---")
display(df_sum)


In [ ]:
# Plot REAL Verified ROC and PR Curves
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")

cases = [
    ("seen_all", "In-Distribution (seen_all)", "#2e7d32"),
    ("unseen_Midjourney_v6", "Unseen Generator (unseen_Midjourney_v6)", "#0288d1"),
    ("gan_to_diffusion", "Cross-Paradigm (gan_to_diffusion)", "#d32f2f")
]

# 1. ROC Curves
plt.figure(figsize=(7.5, 6))
for s, label, color in cases:
    p = f"scores/resnet50_finetuned__{s}.npz"
    if os.path.exists(p):
        d = np.load(p)
        fpr, tpr, _ = roc_curve(d["y_true"], d["y_score"])
        auc = roc_auc_score(d["y_true"], d["y_score"])
        plt.plot(fpr, tpr, label=f"{label} — AUROC = {auc:.4f}", color=color, linewidth=2.5)
plt.plot([0, 1], [0, 1], "--", color="#757575", label="Random Chance (AUROC = 0.5000)")
plt.xlabel("False Positive Rate (FPR)", fontweight="bold")
plt.ylabel("True Positive Rate (TPR)", fontweight="bold")
plt.title("Verified ROC Curves (ResNet-50 Fine-tuned)", fontweight="bold", pad=12)
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig("figures/figure3_real_roc_curves.png", dpi=300)
plt.show()

# 2. Precision-Recall Curves
plt.figure(figsize=(7.5, 6))
for s, label, color in cases:
    p = f"scores/resnet50_finetuned__{s}.npz"
    if os.path.exists(p):
        d = np.load(p)
        prec, rec, _ = precision_recall_curve(d["y_true"], d["y_score"])
        ap = average_precision_score(d["y_true"], d["y_score"])
        plt.plot(rec, prec, label=f"{label} — AP = {ap:.4f}", color=color, linewidth=2.5)
plt.axhline(0.5, linestyle="--", color="#757575", label="Chance Baseline (Prevalence = 0.50)")
plt.xlabel("Recall", fontweight="bold")
plt.ylabel("Precision", fontweight="bold")
plt.title("Verified Precision-Recall Curves (ResNet-50 Fine-tuned)", fontweight="bold", pad=12)
plt.legend(loc="lower left")
plt.tight_layout()
plt.savefig("figures/figure6_real_precision_recall_curves.png", dpi=300)
plt.show()
print("Generated figures/figure3_real_roc_curves.png and figures/figure6_real_precision_recall_curves.png")
